In [51]:
from model import prepare

data = prepare(table)
print(data.groupby("race_id")["won"].sum().value_counts())

won
1    414
Name: count, dtype: int64


## Data notes: missing pole-sitters

In 3 of 329 test races (2010-2025) the qualifying pole-sitter has no row in the cleaned table:
- 2021 R5: DNS (never started)
- 2023 R18 and 2025 R22: started from pole, disqualified after the race (DSQ rows are dropped)

Treatment: the headline pole win rate uses all 329 races, counting these as non-wins (173/329 = 52.6%). Robustness check later: refit without these 3 races.

In [48]:
df[
    df["race_id"].isin(missing)
    & ((df["qualification_position_number"] == 1) | df["grid_position_number"].isna())
][["year", "round", "driver_id", "qualification_position_number",
   "grid_position_number", "grid_position_text", "position_text"]]

,year,round,driver_id,qualification_position_number,grid_position_number,grid_position_text,position_text
6214,2021,5,charles-leclerc,1.0,NaN,NaN,DNS
7341,2023,18,lance-stroll,19.0,NaN,PL,7
7345,2023,18,nico-hulkenberg,16.0,NaN,PL,11
7348,2023,18,kevin-magnussen,14.0,NaN,PL,14
7350,2023,18,fernando-alonso,17.0,NaN,PL,DNF
7354,2023,18,charles-leclerc,1.0,1.0,1,DSQ
8345,2025,22,yuki-tsunoda,19.0,NaN,PL,12
8352,2025,22,lando-norris,1.0,1.0,1,DSQ


In [47]:
has_pole = test.groupby("race_id")["grid"].apply(lambda s: (s == 1).any())
missing = has_pole[~has_pole].index

df[df["race_id"].isin(missing) & (df["grid_position_number"] == 1)][
    ["year", "round", "driver_id", "grid_position_number", "position_text"]
]

,year,round,driver_id,grid_position_number,position_text
7354,2023,18,charles-leclerc,1.0,DSQ
8352,2025,22,lando-norris,1.0,DSQ


In [46]:
print((test["grid"] == 1).sum(), (test["qualification_position_number"] == 1).sum())

326 326


In [45]:
test = table[table["year"].between(2010, 2025)]
print(test["race_id"].nunique())
print(test.loc[test["grid"] == 1, "position_number"].eq(1).mean())
print(test.loc[test["qualification_position_number"] == 1, "position_number"].eq(1).mean())

329
0.5276073619631901
0.5306748466257669


In [44]:
from clean import build_table

table = build_table(df, chron)
print(table.shape, table.duplicated(["race_id", "driver_id"]).sum())
print(table[["grid", "team_form", "driver_rel"]].isna().sum())

(8636, 21) 0
grid            0
team_form      32
driver_rel    105
dtype: int64


In [43]:
from clean import add_driver_rel

df6 = add_driver_rel(df5)
print(df6["driver_rel"].isna().sum(), df6["driver_id"].nunique())
df6[df6["driver_id"] == "lewis-hamilton"].head(3)[
    ["year", "round", "teammate_gap", "driver_rel"]
]

105 103


,year,round,teammate_gap,driver_rel
398,2007,1,-0.045455,NaN
419,2007,2,-0.045455,-0.045455
441,2007,3,0.136364,-0.045455


In [42]:
from clean import add_teammate_gap

df5 = add_teammate_gap(df4)
pairs = df5.groupby(["race_id", "constructor_id"])["teammate_gap"]
print(pairs.sum().abs().max())
print(df5["teammate_gap"].isna().sum())
df5[(df5["year"] == 2024) & (df5["round"] == 1) & (df5["constructor_id"] == "red-bull")][
    ["driver_id", "finish_score", "teammate_gap"]
]

2.220446049250313e-16
74


,driver_id,finish_score,teammate_gap
7435,max-verstappen,1.00,0.05
7436,sergio-perez,0.95,-0.05


In [41]:
from clean import add_team_form

teams3 = add_team_form(teams2)
print(teams3["team_form"].isna().sum())
teams3[teams3["team"] == "ferrari"].head(3)[["year", "round", "team_score", "team_form"]]

16


,year,round,team_score,team_form
1,2006,1,0.795455,NaN
12,2006,2,0.795455,0.795455
23,2006,3,0.000000,0.795455


In [40]:
from clean import lineage_labels, add_lineage

labels = lineage_labels(chron)
teams2 = add_lineage(teams, labels)

print(teams2["constructor_id"].nunique(), teams2["team"].nunique())
print(teams2.duplicated(["race_id", "team"]).sum())

34 16
0


In [39]:
latest = (
    chron.sort_values("year_from")
    .groupby("constructor_id")["other_constructor_id"]
    .last()
)

print(set(chron["other_constructor_id"]) - set(chron["constructor_id"]))

labelled = chron.assign(label=chron["constructor_id"].map(latest))
print(labelled.groupby("other_constructor_id")["label"].nunique().max())

latest[["lotus-f1", "lotus-racing", "toro-rosso", "honda"]]

set()
1


constructor_id
lotus-f1              alpine
lotus-racing        caterham
toro-rosso      racing-bulls
honda               mercedes
Name: other_constructor_id, dtype: str

In [38]:
lineage = chron[["other_constructor_id", "constructor_id"]].drop_duplicates()
print(lineage["other_constructor_id"].duplicated().sum())

print(set(teams["constructor_id"]) - set(lineage["other_constructor_id"]))

lineage[lineage["other_constructor_id"].str.startswith("lotus")]

154
{'haas', 'mclaren', 'cadillac', 'toyota', 'hrt', 'ferrari', 'williams', 'super-aguri'}


,other_constructor_id,constructor_id
14,lotus-f1,alpine
42,lotus-f1,benetton
56,lotus-racing,caterham
109,lotus-f1,lotus-f1
112,lotus-racing,lotus-racing
170,lotus-f1,renault
191,lotus-f1,toleman


In [37]:
chron = pd.read_sql("SELECT * FROM constructor_chronology", con)
print(chron.shape)
chron.head(10)

(217, 5)


,constructor_id,position_display_order,other_constructor_id,year_from,year_to
0,alfa-romeo,1,sauber,1993,2005.0
1,alfa-romeo,2,bmw-sauber,2006,2010.0
2,alfa-romeo,3,sauber,2011,2018.0
3,alfa-romeo,4,alfa-romeo,2019,2023.0
4,alfa-romeo,5,kick-sauber,2024,2025.0
5,alfa-romeo,6,audi,2026,NaN
6,alphatauri,1,minardi,1985,2005.0
7,alphatauri,2,toro-rosso,2006,2019.0
8,alphatauri,3,alphatauri,2020,2023.0
9,alphatauri,4,rb,2024,2024.0


In [36]:
span = teams.groupby("constructor_id")["year"].agg(["min", "max", "count"])
span.sort_values("min")

,min,max,count
constructor_id,,,
williams,2006,2026,414
toyota,2006,2009,70
toro-rosso,2006,2019,268
super-aguri,2006,2008,39
bmw-sauber,2006,2009,70
renault,2006,2020,207
ferrari,2006,2026,413
red-bull,2006,2026,414
honda,2006,2008,53


In [35]:
from clean import team_race_scores

teams = team_race_scores(df4)
print(teams.shape)
teams[(teams["year"] == 2024) & (teams["round"] == 1)].head(3)

(4355, 5)


,race_id,year,round,constructor_id,team_score
3713,1102,2024,1,alpine,0.175
3714,1102,2024,1,aston-martin,0.575
3715,1102,2024,1,ferrari,0.875


In [34]:
from clean import add_finish_score

df4 = add_finish_score(df3)
print(df4.loc[df4["finish_score"] == 0, "position_text"].value_counts())
print(df4.groupby("race_id")["finish_score"].max().eq(1.0).all())

position_text
DNF    1412
NC       10
Name: count, dtype: int64
True


In [32]:
from clean import flag_sprint_grid

df3 = flag_sprint_grid(fill_pit_lane_grid(drop_non_starters(df)))
print(df3["grid"].isna().sum())
print(df3.loc[df3["is_sprint_grid"], "race_id"].nunique())

0
6


In [30]:
print(df2["grid"].isna().sum())
print(df2["race_id"].dtype)

df2[
    (df2["driver_id"] == "pierre-gasly")
    & (df2["year"] == 2021)
    & (df2["round"] == 14)
][["driver_id", "grid_position_number", "grid"]]

0
int64


,driver_id,grid_position_number,grid
6393,pierre-gasly,NaN,20.0


In [29]:
print(df2["grid"].isna().sum())

df2[df2["grid_position_number"].isna()][
    ["race_id", "driver_id", "grid_position_number", "grid"]
].head(10)

0


,race_id,driver_id,grid_position_number,grid
16,751,tiago-monteiro,NaN,23.0
48,753,giancarlo-fisichella,NaN,23.0
136,757,michael-schumacher,NaN,22.0
201,760,jarno-trulli,NaN,22.0
260,762,sakon-yamamoto,NaN,21.0
520,774,christijan-albers,NaN,23.0
576,777,felipe-massa,NaN,22.0
585,777,takuma-sato,NaN,23.0
712,783,vitantonio-liuzzi,NaN,23.0
785,786,adrian-sutil,NaN,22.0


In [28]:
from clean import fill_pit_lane_grid

df2 = fill_pit_lane_grid(df1)
print(df2["grid"].isna().sum())
df2[df2["driver_id"].eq("pierre_gasly") & df2["race_id"].eq(1049)][
    ["driver_id", "grid_position_number", "grid"]
]

0


,driver_id,grid_position_number,grid


In [27]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")
from clean import drop_non_starters

df1 = drop_non_starters(df)
len(df), len(df1)

(8724, 8636)

## Data notes: sprint-format races

In 2021-22 (6 races, `qualifying_format = SPRINT_RACE`) the Sunday grid was set by the Saturday sprint result, not by qualifying. Check: only 55% of drivers started where they qualified, vs 72% in normal 2021-22 races.

From 2023 the Sunday grid always comes from a qualifying session, so those weekends are unaffected.

Treatment: keep the 6 races and flag them (`is_sprint_grid`). Robustness check later: refit without them. Exclude them from the qualifying-gap and penalty analysis.

In [25]:
pd.read_sql("""
SELECT r.qualifying_format,
    COUNT(*) AS drivers,
    SUM(rr.grid_position_number = rr.qualification_position_number) AS same
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2021 AND 2022
AND rr.grid_position_number IS NOT NULL
AND rr.qualification_position_number IS NOT NULL
GROUP BY r.qualifying_format
""", con)

,qualifying_format,drivers,same
0,KNOCKOUT,733,527
1,SPRINT_RACE,112,62


In [23]:
pd.read_sql("""
SELECT year, qualifying_format, sprint_qualifying_format,
       COUNT(*) AS races
FROM race
WHERE year BETWEEN 2006 AND 2026
GROUP BY year, qualifying_format, sprint_qualifying_format
ORDER BY year
""", con)

,year,qualifying_format,sprint_qualifying_format,races
0,2006,KNOCKOUT,NaN,18
1,2007,KNOCKOUT,NaN,17
2,2008,KNOCKOUT,NaN,18
3,2009,KNOCKOUT,NaN,17
4,2010,KNOCKOUT,NaN,19
5,2011,KNOCKOUT,NaN,19
6,2012,KNOCKOUT,NaN,20
7,2013,KNOCKOUT,NaN,19
8,2014,KNOCKOUT,NaN,19
9,2015,KNOCKOUT,NaN,19


In [22]:
pd.read_sql("""
SELECT year, round, official_name, qualifying_format, sprint_qualifying_format
FROM race
WHERE id = 1049
""", con)

,year,round,official_name,qualifying_format,sprint_qualifying_format
0,2021,14,Formula 1 Heineken Gran Premio d'Italia 2021,SPRINT_RACE,None


In [20]:
no_grid[no_grid["position_text"] == "DNF"]

,race_id,year,round,date,circuit_id,driver_id,constructor_id,qualification_position_number,qualification_position_text,grid_position_number,grid_position_text,position_number,position_text
6393,1049,2021,14,2021-09-12,monza,pierre-gasly,alphatauri,NaN,DNF,NaN,NaN,NaN,DNF


In [19]:
no_grid["position_text"].value_counts()

position_text
DNS    16
DNQ     2
DNP     2
DNF     1
Name: count, dtype: int64

In [18]:
no_grid = df[df["grid_position_text"].isna()]
no_grid["position_number"].notna().value_counts()

position_number
False    21
Name: count, dtype: int64

In [31]:
from pathlib import Path

query = Path("../sql/01_race_results.sql").read_text()
df = pd.read_sql(query, con)
df.shape

(8724, 14)

## Data notes: grid positions

Blank `grid_position_number` values (2006-2025, 191 rows):

| Grid label | Meaning | Treatment |
|---|---|---|
| PL | Pit-lane start | Ranked behind the last gridded car, ordered by qualifying position |
| NaN | No grid information | Check whether the driver finished; drop only if they never started |

Missing qualifying positions (NC, EX, DSQ) don't affect the main model, which uses the grid slot. They matter later for the qualifying-gap and penalty analysis.

In [12]:
query = """
SELECT rr.grid_position_text, rr.qualification_position_text,
    COUNT(*) AS n
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2025
AND (rr.grid_position_number IS NULL
    OR rr.qualification_position_number IS NULL)
GROUP BY rr.grid_position_text, rr.qualification_position_text
ORDER BY n DESC
"""

pd.read_sql(query, con)

,grid_position_text,qualification_position_text,n
0,20,NC,26
1,PL,19,26
2,PL,20,24
3,PL,18,18
4,PL,NC,15
5,19,NC,14
6,22,NC,10
7,PL,14,9
8,PL,16,9
9,PL,10,8


In [9]:
query = """
SELECT r.year,
    COUNT(DISTINCT r.id) AS races,
    COUNT(*) AS rows,
    COUNT(*) - COUNT(rr.grid_position_number) AS no_grid,
    COUNT(*) - COUNT(rr.qualification_position_number) AS no_qual
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2026
GROUP BY r.year
ORDER BY r.year
"""

pd.read_sql(query, con)

,year,races,rows,no_grid,no_qual
0,2006,18,396,5,2
1,2007,17,374,4,0
2,2008,18,368,11,1
3,2009,17,340,9,3
4,2010,19,456,9,1
5,2011,19,454,2,14
6,2012,20,480,6,10
7,2013,19,418,1,5
8,2014,19,407,14,12
9,2015,19,380,7,12


## Data notes: non-finishers

Every non-finish has a blank `position_number`, with the reason in `position_text` (2006-2025 counts):

| Label | Count | Treatment |
|---|---|---|
| DNF | 1,355 | Counts as 0 in team form |
| NC | 8 | Counts as 0 in team form |
| DNS | 49 | Row dropped (never raced) |
| DSQ | 28 | Row dropped (rules breach, not pace) |
| DNQ / DNP | 2 / 2 | Row dropped (never raced) |

Caveat: DNF mixes crashes and mechanical failures, so team form partly reflects reliability, not just speed.

In [7]:
query = """
SELECT rr.position_text, COUNT(*) AS n
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2025
AND rr.position_number IS NULL
GROUP BY rr.position_text
ORDER BY n DESC
"""

pd.read_sql(query, con)

,position_text,n
0,DNF,1355
1,DNS,49
2,DSQ,28
3,NC,8
4,DNQ,2
5,DNP,2


In [5]:
query = """
SELECT r.year, r.round, rr.driver_id,
       rr.qualification_position_number AS qual,
       rr.grid_position_number AS grid,
       rr.position_number, rr.position_text, rr.reason_retired
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year = 2024
AND (rr.position_number IS NULL
    OR rr.qualification_position_number != rr.grid_position_number)
ORDER BY r.round
LIMIT 15
"""

pd.read_sql(query, con)

,year,round,driver_id,qual,grid,position_number,position_text,reason_retired
0,2024,2,lance-stroll,10,10,NaN,DNF,Accident
1,2024,2,pierre-gasly,18,18,NaN,DNF,Gearbox
2,2024,3,charles-leclerc,5,4,2.0,2,NaN
3,2024,3,lando-norris,4,3,3.0,3,NaN
4,2024,3,oscar-piastri,6,5,4.0,4,NaN
5,2024,3,sergio-perez,3,6,5.0,5,NaN
6,2024,3,lewis-hamilton,11,11,NaN,DNF,Power unit
7,2024,3,max-verstappen,1,1,NaN,DNF,Brakes
8,2024,4,guanyu-zhou,20,20,NaN,DNF,Gearbox
9,2024,4,daniel-ricciardo,11,11,NaN,DNF,Collision


In [3]:
query = """
SELECT r.year, r.round, rr.driver_id,rr.constructor_id,
       rr.qualification_position_number, rr.grid_position_number,
       rr.position_number, rr.position_text
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year = 2024 AND r.round = 1
ORDER BY rr.position_display_order
"""

pd.read_sql(query, con)

,year,round,driver_id,constructor_id,qualification_position_number,grid_position_number,position_number,position_text
0,2024,1,max-verstappen,red-bull,1,1,1,1
1,2024,1,sergio-perez,red-bull,5,5,2,2
2,2024,1,carlos-sainz-jr,ferrari,4,4,3,3
3,2024,1,charles-leclerc,ferrari,2,2,4,4
4,2024,1,george-russell,mercedes,3,3,5,5
5,2024,1,lando-norris,mclaren,7,7,6,6
6,2024,1,lewis-hamilton,mercedes,9,9,7,7
7,2024,1,oscar-piastri,mclaren,8,8,8,8
8,2024,1,fernando-alonso,aston-martin,6,6,9,9
9,2024,1,lance-stroll,aston-martin,12,12,10,10


In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/raw/f1db.db")